# Prompt Engineering and Evaluation with Gemini

This notebook demonstrates how to build a robust prompt evaluation system using the Gemini API. We will walk through generating synthetic test datasets, running evaluations with custom criteria, and visualizing the results in an interactive HTML report.

## 1. Imports and Environment Setup

We start by importing the necessary libraries and loading our environment variables, specifically the `GOOGLE_API_KEY`.

In [ ]:
import os
import json
import concurrent.futures
import re
import sys
from pathlib import Path
from textwrap import dedent
from statistics import mean
from dotenv import load_dotenv
from google import genai
from google.genai import types
from IPython.display import display, Markdown, HTML

project_root = next(
    (path for path in [Path.cwd(), *Path.cwd().parents] if (path / "requirements.txt").exists()),
    Path.cwd(),
)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from utils.gemini_retry import generate_content_with_retry

## 2. Client Initialization and Helper Functions

Here we initialize the Gemini client and define a helper `chat` function. This function abstracts the Gemini SDK's `generate_content` method to make it easy to send messages and receive responses.

Gemini calls in this notebook now use the shared Tenacity-based retry helper for `429` and `RESOURCE_EXHAUSTED` responses.

In [ ]:
load_dotenv()
api_key = os.getenv("GOOGLE_API_KEY")
client = genai.Client(api_key=api_key)
model = "gemini-2.5-flash"

def add_user_message(messages, text):
    messages.append({"role": "user", "content": text})

def add_assistant_message(messages, text):
    messages.append({"role": "model", "content": text})

def chat(messages, system=None, temperature=1.0, stop_sequences=[], response_mime_type=None, max_retries=5):
    contents = []
    for m in messages:
        role = "user" if m["role"] == "user" else "model"
        contents.append({"role": role, "parts": [{"text": m["content"]}]})
    
    config = types.GenerateContentConfig(
        temperature=temperature,
        max_output_tokens=2048,
        stop_sequences=stop_sequences,
        system_instruction=system,
        response_mime_type=response_mime_type
    )
    
    response = generate_content_with_retry(
        client=client,
        model=model,
        contents=contents,
        config=config,
        max_attempts=max_retries,
    )
    return response.text

## 3. HTML Report Generation

To make evaluation results digestible, we use this function to generate a styled HTML report. It summarizes the average score, pass rate, and provides a detailed breakdown of every test case.

In [3]:
def generate_prompt_evaluation_report(evaluation_results):
    total_tests = len(evaluation_results)
    scores = [result["score"] for result in evaluation_results]
    avg_score = mean(scores) if scores else 0
    max_possible_score = 10
    pass_rate = (
        100 * len([s for s in scores if s >= 7]) / total_tests if total_tests else 0
    )

    html = f"""
    <!DOCTYPE html>
    <html lang="en">
    <head>
        <meta charset="UTF-8">
        <meta name="viewport" content="width=device-width, initial-scale=1.0">
        <title>Prompt Evaluation Report</title>
        <style>
            body {{ font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, Helvetica, Arial, sans-serif; line-height: 1.6; padding: 20px; color: #333; max-width: 1200px; margin: 0 auto; }}
            .header {{ background-color: #f8f9fa; padding: 30px; border-radius: 12px; margin-bottom: 30px; border: 1px solid #e1e4e8; }}
            .summary-stats {{ display: flex; justify-content: space-between; flex-wrap: wrap; gap: 20px; }}
            .stat-box {{ background-color: #fff; border-radius: 8px; padding: 20px; box-shadow: 0 4px 6px rgba(0,0,0,0.05); flex: 1; min-width: 200px; border: 1px solid #eee; }}
            .stat-value {{ font-size: 28px; font-weight: bold; margin-top: 10px; color: #1a73e8; }}
            table {{ width: 100%; border-collapse: collapse; margin-top: 30px; background: #fff; border-radius: 8px; overflow: hidden; box-shadow: 0 4px 6px rgba(0,0,0,0.05); }}
            th {{ background-color: #3c4043; color: white; text-align: left; padding: 15px; font-weight: 500; }}
            td {{ padding: 15px; border-bottom: 1px solid #eee; vertical-align: top; }}
            tr:hover {{ background-color: #fcfcfc; }}
            .score {{ font-weight: bold; padding: 6px 12px; border-radius: 20px; display: inline-block; font-size: 14px; }}
            .score-high {{ background-color: #e6f4ea; color: #137333; }}
            .score-medium {{ background-color: #fef7e0; color: #b06000; }}
            .score-low {{ background-color: #fce8e6; color: #c5221f; }}
            .output pre {{ background-color: #f8f9fa; padding: 12px; border-radius: 6px; border: 1px solid #e1e4e8; font-family: 'Roboto Mono', monospace; font-size: 13px; white-space: pre-wrap; margin: 0; }}
        </style>
    </head>
    <body>
        <div class="header">
            <h1>Prompt Evaluation Report</h1>
            <div class="summary-stats">
                <div class="stat-box"><div>Total Test Cases</div><div class="stat-value">{total_tests}</div></div>
                <div class="stat-box"><div>Average Score</div><div class="stat-value">{avg_score:.1f} / {max_possible_score}</div></div>
                <div class="stat-box"><div>Pass Rate (≥7)</div><div class="stat-value">{pass_rate:.1f}%</div></div>
            </div>
        </div>
        <table>
            <thead><tr><th>Scenario</th><th>Inputs</th><th>Criteria</th><th>Output</th><th>Score</th><th>Reasoning</th></tr></thead>
            <tbody>
    """

    for result in evaluation_results:
        inputs_html = "<br>".join([f"<b>{k}:</b> {v}" for k, v in result["test_case"]["prompt_inputs"].items()])
        criteria_html = "<ul>" + "".join([f"<li>{c}</li>" for c in result["test_case"]["solution_criteria"]]) + "</ul>"
        
        s = result["score"]
        s_cls = "score-high" if s >= 8 else ("score-low" if s <= 5 else "score-medium")
        
        html += f"""
            <tr>
                <td>{result['test_case']['scenario']}</td>
                <td>{inputs_html}</td>
                <td>{criteria_html}</td>
                <td class='output'><pre>{result['output']}</pre></td>
                <td><span class='score {s_cls}'>{s}</span></td>
                <td>{result['reasoning']}</td>
            </tr>
        """
    html += "</tbody></table></body></html>"
    return html

## 4. PromptEvaluator Class Implementation

The `PromptEvaluator` class handles the core logic of our system:
1.  **Idea Generation:** Brainstorms diverse scenarios for testing.
2.  **Test Case Creation:** Generates detailed inputs and criteria for each scenario.
3.  **Grading:** Evaluates the target prompt's output against the generated criteria using Gemini's JSON mode.

In [4]:
class PromptEvaluator:
    def __init__(self, max_concurrent_tasks=3):
        self.max_concurrent_tasks = max_concurrent_tasks

    def render(self, template_string, variables):
        for key, value in variables.items():
            template_string = template_string.replace("{" + key + "}", str(value))
        return template_string

    def generate_unique_ideas(self, task_description, prompt_inputs_spec, num_cases):
        prompt = f"""
        Generate {num_cases} unique, diverse ideas for testing a prompt that accomplishes this task:
        <task_description>{task_description}</task_description>
        The prompt receives these inputs: {prompt_inputs_spec}
        Output a JSON array of strings, where each string is a brief test scenario idea.
        """
        system = "You are a test scenario designer. Output ONLY valid JSON."
        text = chat([{"role": "user", "content": prompt}], system=system, response_mime_type="application/json")
        return json.loads(text)

    def generate_test_case(self, task_description, idea, prompt_inputs_spec):
        prompt = f"""
        Create a detailed test case for:
        Task: {task_description}
        Scenario Idea: {idea}
        Available Input Keys: {list(prompt_inputs_spec.keys())}
        
        Return JSON with:
        - "prompt_inputs": map of input keys to specific test values
        - "solution_criteria": list of 1-4 measurable criteria
        """
        system = "You are an expert test engineer. Output ONLY valid JSON."
        text = chat([{"role": "user", "content": prompt}], system=system, response_mime_type="application/json")
        test_case = json.loads(text)
        test_case.update({"task_description": task_description, "scenario": idea})
        return test_case

    def generate_dataset(self, task_description, prompt_inputs_spec, num_cases=1, output_file="dataset.json"):
        ideas = self.generate_unique_ideas(task_description, prompt_inputs_spec, num_cases)
        dataset = []
        with concurrent.futures.ThreadPoolExecutor(max_workers=self.max_concurrent_tasks) as executor:
            futures = [executor.submit(self.generate_test_case, task_description, idea, prompt_inputs_spec) for idea in ideas]
            for f in concurrent.futures.as_completed(futures):
                dataset.append(f.result())
                print(f"Generated {len(dataset)}/{len(ideas)} test cases")
        with open(output_file, "w") as f: json.dump(dataset, f, indent=2)
        return dataset

    def grade_output(self, test_case, output, extra_criteria):
        prompt = f"""
        Evaluate this AI output with RIGOR.
        Task: {test_case['task_description']}
        Inputs: {test_case['prompt_inputs']}
        Output to Grade: {output}
        Criteria: {test_case['solution_criteria']}
        Additional Constraints: {extra_criteria}
        
        Return JSON with fields: "strengths" (list), "weaknesses" (list), "reasoning" (string), "score" (1-10).
        """
        system = "You are a strict evaluator. Output ONLY valid JSON."
        text = chat([{"role": "user", "content": prompt}], system=system, temperature=0.0, response_mime_type="application/json")
        return json.loads(text)

    def run_evaluation(self, run_prompt_fn, dataset_file, extra_criteria=None, html_file="output.html"):
        with open(dataset_file, "r") as f: dataset = json.load(f)
        results = []
        for tc in dataset:
            output = run_prompt_fn(tc["prompt_inputs"])
            grade = self.grade_output(tc, output, extra_criteria)
            results.append({"output": output, "test_case": tc, "score": grade["score"], "reasoning": grade["reasoning"]})
            print(f"Graded {len(results)}/{len(dataset)} cases")
        
        avg = mean([r["score"] for r in results])
        print(f"Average Score: {avg:.2f}")
        html = generate_prompt_evaluation_report(results)
        with open(html_file, "w", encoding="utf-8") as f: f.write(html)
        display(HTML(f"<h3>Evaluation Complete!</h3><p>Average Score: {avg:.2f}</p><a href='{html_file}' target='_blank'>View Full Report</a>"))
        return results

## 5. Initialize the Evaluator

We create an instance of the `PromptEvaluator`. We'll keep concurrency low to avoid hitting rate limits while generating the dataset.

In [ ]:
evaluator = PromptEvaluator(max_concurrent_tasks=2)

## 6. Generate the Test Dataset

We define our task (e.g., generating meal plans) and the expected inputs. The evaluator then generates diverse test scenarios automatically.

In [6]:
dataset = evaluator.generate_dataset(
    task_description="Write a compact, concise 1 day meal plan for a single athlete",
    prompt_inputs_spec={
        "height": "Athlete's height in cm",
        "weight": "Athlete's weight in kg",
        "goal": "Goal of the athlete",
        "restrictions": "Dietary restrictions of the athlete",
    },
    num_cases=3,
    output_file="../../data/dataset_gemini.json"
)

Rate limit or error hit, retrying in 2.5s... (Attempt 2/5)
Rate limit or error hit, retrying in 5.0s... (Attempt 3/5)
Rate limit or error hit, retrying in 9.5s... (Attempt 4/5)
Rate limit or error hit, retrying in 18.0s... (Attempt 5/5)


ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash\nPlease retry in 1.689218415s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTier', 'quotaDimensions': {'location': 'global', 'model': 'gemini-2.5-flash'}, 

## 7. Define the Prompt to be Evaluated

This function contains the actual prompt we want to test. It takes the synthetic inputs from our dataset and returns the model's response.

In [8]:
def run_prompt(inputs):
    prompt = f"""
    Generate a one-day meal plan for an athlete.
    Info: {inputs['height']}cm, {inputs['weight']}kg. 
    Goal: {inputs['goal']}. Restrictions: {inputs['restrictions']}.
    
    Requirements:
    1. Calorie total
    2. Macro breakdown (P/F/C)
    3. Meal timing
    4. Portion sizes in grams
    """
    return chat([{"role": "user", "content": prompt}])

## 8. Run the Full Evaluation

Finally, we run the evaluation. This executes the prompt for every test case, grades the results, and produces the HTML report.

In [ ]:
results = evaluator.run_evaluation(
    run_prompt_fn=run_prompt,
    dataset_file="../../data/dataset_gemini.json",
    extra_criteria="Ensure the meal plan is actually high-protein for an athlete.",
    html_file="../../reports/prompt_eval_report.html"
)

Graded 1/3 cases
Graded 2/3 cases
Graded 3/3 cases
Average Score: 1.00
